# Model Performance vs Human Annotations

Evaluates four label sources against 100 manually annotated cases (`data/random_annotations.csv`):
- **CT-RATE (original)** — dataset-provided labels
- **gpt-5.4-mini** — GPT-5.4-mini LLM labels
- **gpt-5.4** — GPT-5.4 LLM labels
- **DeepSeek-V3.2** — DeepSeek-V3.2 LLM labels

Metrics computed per label and source (treating human annotations as ground truth):
- **Sensitivity** — TP / (TP + FN)
- **Specificity** — TN / (TN + FP)
- **PPV** — TP / (TP + FP)
- **NPV** — TN / (TN + FN)
- **Accuracy** — (TP + TN) / N
- **F1** — binary F1 for the positive class
- **Cohen's κ** — chance-corrected agreement

In the aggregate summary (Table 2) two averaging strategies are shown:
- **Label-macro** — simple mean of per-label metrics across 18 labels
- **Micro** — TP / FP / FN pooled across all 18 labels, then metrics computed once

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import confusion_matrix, cohen_kappa_score, f1_score

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'figure.dpi': 150,
})

DATA_DIR    = Path('../data')
FIGURES_DIR = Path('../figures')
TABLES_DIR  = Path('../tables')

LABEL_COLS = [
    'Medical material',
    'Arterial wall calcification',
    'Cardiomegaly',
    'Pericardial effusion',
    'Coronary artery wall calcification',
    'Hiatal hernia',
    'Lymphadenopathy',
    'Emphysema',
    'Atelectasis',
    'Lung nodule',
    'Lung opacity',
    'Pulmonary fibrotic sequela',
    'Pleural effusion',
    'Mosaic attenuation pattern',
    'Peribronchial thickening',
    'Consolidation',
    'Bronchiectasis',
    'Interlobular septal thickening',
]

LLM_MODELS  = ['GPT-5.4', 'GPT-5.4 mini', 'DeepSeek-V3.2']
VOTING_NAME = 'Majority vote (3 LLMs)'

SOURCE_ORDER   = ['CT-RATE (original)'] + LLM_MODELS + [VOTING_NAME]
_colors        = plt.cm.tab10(np.linspace(0, 0.5, len(SOURCE_ORDER)))
SOURCE_COLORS  = dict(zip(SOURCE_ORDER, _colors))
SOURCE_MARKERS = dict(zip(SOURCE_ORDER, ['s', 'o', '^', 'D', 'P']))

## 1. Load Human Annotations

In [ ]:
ann_long = pd.read_csv(DATA_DIR / 'random_annotations_v2.csv')
ann_long = ann_long[ann_long["annotator"]=="髙松 篤"].reset_index(drop=True)


# Pivot long → wide: one row per volume, one column per label
human = (
    ann_long
    .pivot_table(index='volume_name', columns='label_name', values='value', aggfunc='first')
    .reset_index()
    .rename(columns={'volume_name': 'VolumeName'})
)
human = human[['VolumeName'] + LABEL_COLS]

annotated_volumes = set(human['VolumeName'])

print(f'Human-annotated volumes : {len(human):,}')
print(f'Labels                  : {len(LABEL_COLS)}')
print(f'Overall positive rate   : {human[LABEL_COLS].values.mean():.1%}')
print()
print('Per-label positive rate (human ground truth):')
print((human[LABEL_COLS].mean() * 100).round(1).rename('Prevalence (%)').to_string())

## 2. Load Model Predictions and CT-RATE Original Labels

In [ ]:
SOURCE_FILES = {
    'CT-RATE (original)': DATA_DIR / 'CT-RATE_reports.csv',
    'GPT-5.4':            DATA_DIR / 'CT-RATE_reports_gpt-5.4.csv',
    'GPT-5.4 mini':       DATA_DIR / 'CT-RATE_reports_gpt-5.4-mini.csv',
    'DeepSeek-V3.2':      DATA_DIR / 'CT-RATE_reports_DeepSeek-V3.2.csv',
}


def load_source(path: Path, name: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    if '_extraction_failed' in df.columns:
        n_before = len(df)
        df = df[~df['_extraction_failed'].fillna(True).astype(bool)].copy()
        print(f'  {name}: dropped {n_before - len(df):,} extraction failures')
    df = df[df['VolumeName'].isin(annotated_volumes)][['VolumeName'] + LABEL_COLS].copy()
    missing = annotated_volumes - set(df['VolumeName'])
    if missing:
        print(f'  {name}: WARNING — {len(missing)} annotated volumes missing')
    print(f'  {name}: {len(df):,} rows matched')
    return df


print('Loading sources:')
source_dfs = {name: load_source(path, name) for name, path in SOURCE_FILES.items()}

# Find volumes present in ALL sources AND human annotations
common_volumes = annotated_volumes.copy()
for df in source_dfs.values():
    common_volumes &= set(df['VolumeName'])
print(f'\nVolumes in all sources + human annotations: {len(common_volumes):,}')

# Use a single sorted index so all frames are row-aligned
SORTED_IDX = sorted(common_volumes)

human_idx = human.set_index('VolumeName').loc[SORTED_IDX]
source_idx = {
    name: df.set_index('VolumeName').loc[SORTED_IDX]
    for name, df in source_dfs.items()
}

# --- Majority vote across 3 LLMs (threshold ≥ 2 out of 3) ---
vote_df = pd.DataFrame(index=SORTED_IDX)
for col in LABEL_COLS:
    votes = np.stack([source_idx[m][col].astype(int).values for m in LLM_MODELS], axis=1)
    vote_df[col] = (votes.sum(axis=1) >= 2).astype(int)
source_idx[VOTING_NAME] = vote_df

print(f'\nMajority vote added (≥2/3 models agree on positive)')
print('Row alignment verified:', all(
    (src.index == human_idx.index).all() for src in source_idx.values()
))

## 3. Per-Label Performance Metrics

In [ ]:
def wilson_ci(k, n, alpha=0.05):
    """Wilson score 95% CI for proportion k/n."""
    if n == 0:
        return np.nan, np.nan
    z = stats.norm.ppf(1 - alpha / 2)
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return float(max(0.0, center - margin)), float(min(1.0, center + margin))


def kappa_ci(y_true, y_pred, alpha=0.05):
    """Large-sample normal approximation CI for Cohen's κ."""
    n = len(y_true)
    try:
        kappa = cohen_kappa_score(y_true, y_pred)
    except Exception:
        return np.nan, np.nan
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    p_o = (tp + tn) / n
    p_e = ((tp + fn) / n) * ((tp + fp) / n) + ((tn + fp) / n) * ((tn + fn) / n)
    if (1 - p_e) == 0:
        return np.nan, np.nan
    se = np.sqrt(p_o * (1 - p_o) / (n * (1 - p_e) ** 2))
    z = stats.norm.ppf(1 - alpha / 2)
    return float(kappa - z * se), float(kappa + z * se)


def fmt_pct(val, lo=None, hi=None):
    """Format proportion as 'XX.X% (XX.X–XX.X)'."""
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return 'N/A'
    s = f'{val * 100:.1f}%'
    if lo is not None and hi is not None and not np.isnan(lo) and not np.isnan(hi):
        s += f' ({lo * 100:.1f}–{hi * 100:.1f})'
    return s


def fmt_kappa(val, lo=None, hi=None):
    """Format κ as '0.XXX (0.XXX–0.XXX)'."""
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return 'N/A'
    s = f'{val:.3f}'
    if lo is not None and hi is not None and not np.isnan(lo) and not np.isnan(hi):
        s += f' ({lo:.3f}–{hi:.3f})'
    return s


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    n = len(y_true)

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    ppv         = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    npv         = tn / (tn + fn) if (tn + fn) > 0 else np.nan
    accuracy    = (tp + tn) / n
    f1          = f1_score(y_true, y_pred, average='binary', zero_division=np.nan)
    try:
        kappa = cohen_kappa_score(y_true, y_pred)
    except Exception:
        kappa = np.nan

    sens_lo, sens_hi = wilson_ci(tp, tp + fn)
    spec_lo, spec_hi = wilson_ci(tn, tn + fp)
    ppv_lo,  ppv_hi  = wilson_ci(tp, tp + fp)
    npv_lo,  npv_hi  = wilson_ci(tn, tn + fn)
    acc_lo,  acc_hi  = wilson_ci(tp + tn, n)
    # F1 = 2TP/(2TP+FP+FN): Wilson CI with numerator=2TP, denominator=2TP+FP+FN
    f1_denom = 2 * tp + fp + fn
    f1_lo, f1_hi = (wilson_ci(2 * tp, f1_denom) if f1_denom > 0 else (np.nan, np.nan))
    kap_lo, kap_hi = kappa_ci(y_true, y_pred)

    def r(v):
        return round(float(v), 4) if v is not None and not np.isnan(v) else np.nan

    return {
        'N': n,
        'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn),
        'Prev_human_%':  round(y_true.mean() * 100, 1),
        'Prev_source_%': round(y_pred.mean() * 100, 1),
        'Sensitivity': r(sensitivity),
        'Sensitivity_lo': r(sens_lo), 'Sensitivity_hi': r(sens_hi),
        'Specificity': r(specificity),
        'Specificity_lo': r(spec_lo), 'Specificity_hi': r(spec_hi),
        'PPV':  r(ppv),  'PPV_lo':  r(ppv_lo),  'PPV_hi':  r(ppv_hi),
        'NPV':  r(npv),  'NPV_lo':  r(npv_lo),  'NPV_hi':  r(npv_hi),
        'Accuracy':    r(accuracy),
        'Accuracy_lo': r(acc_lo), 'Accuracy_hi': r(acc_hi),
        'F1':    r(f1),    'F1_lo':    r(f1_lo),    'F1_hi':    r(f1_hi),
        'Kappa': r(kappa), 'Kappa_lo': r(kap_lo),  'Kappa_hi': r(kap_hi),
    }


METRIC_COLS = ['Sensitivity', 'Specificity', 'PPV', 'NPV', 'Accuracy', 'F1', 'Kappa']

rows = []
for source_name in SOURCE_ORDER:
    src_df = source_idx[source_name]
    for col in LABEL_COLS:
        y_true = human_idx[col].astype(int).values
        y_pred = src_df[col].astype(int).values
        rows.append({'Source': source_name, 'Label': col, **compute_metrics(y_true, y_pred)})

metrics_df = pd.DataFrame(rows)
print('Metrics computed. Sources:', metrics_df['Source'].unique().tolist())

In [ ]:
print('Table 1. Per-label metrics vs human annotations\n')
for src in SOURCE_ORDER:
    sub = metrics_df[metrics_df['Source'] == src].set_index('Label')

    disp = pd.DataFrame(index=sub.index)
    disp['N']             = sub['N']
    disp['Prev_human_%']  = sub['Prev_human_%']
    disp['Prev_source_%'] = sub['Prev_source_%']
    for m in ['Sensitivity', 'Specificity', 'PPV', 'NPV', 'Accuracy', 'F1']:
        disp[m] = [
            fmt_pct(sub.loc[lbl, m], sub.loc[lbl, f'{m}_lo'], sub.loc[lbl, f'{m}_hi'])
            for lbl in sub.index
        ]
    disp['Kappa'] = [
        fmt_kappa(sub.loc[lbl, 'Kappa'], sub.loc[lbl, 'Kappa_lo'], sub.loc[lbl, 'Kappa_hi'])
        for lbl in sub.index
    ]

    print(f'--- {src} ---')
    print(disp.to_string())
    print()

## 4. Macro-Averaged Summary

In [ ]:
# --- Label-macro: simple mean of per-label metrics (numeric, kept for figures) ---
label_macro = (
    metrics_df.groupby('Source')[METRIC_COLS]
    .mean()
    .round(4)
    .reindex(SOURCE_ORDER)
)
label_macro.index.name = 'Source'

# 95% CI for macro mean: t-distribution across 18 per-label values
macro_ci = {}
for src in SOURCE_ORDER:
    sub = metrics_df[metrics_df['Source'] == src]
    macro_ci[src] = {}
    for m in METRIC_COLS:
        vals = sub[m].astype(float).dropna().values
        mean_val = vals.mean()
        if len(vals) > 1:
            se = vals.std(ddof=1) / np.sqrt(len(vals))
            t_crit = stats.t.ppf(0.975, df=len(vals) - 1)
            macro_ci[src][m] = (mean_val, mean_val - t_crit * se, mean_val + t_crit * se)
        else:
            macro_ci[src][m] = (mean_val, np.nan, np.nan)

# --- Micro: pool TP/FP/FN across all 18 labels (numeric, kept for figures) ---
micro_rows = []
for src in SOURCE_ORDER:
    src_df = source_idx[src]
    all_true = np.concatenate([human_idx[col].astype(int).values for col in LABEL_COLS])
    all_pred = np.concatenate([src_df[col].astype(int).values    for col in LABEL_COLS])
    cm = confusion_matrix(all_true, all_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    n = len(all_true)

    sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    ppv  = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    npv  = tn / (tn + fn) if (tn + fn) > 0 else np.nan
    acc  = (tp + tn) / n
    f1   = float(f1_score(all_true, all_pred, average='binary', zero_division=np.nan))
    kap  = cohen_kappa_score(all_true, all_pred)

    sens_lo, sens_hi = wilson_ci(tp, tp + fn)
    spec_lo, spec_hi = wilson_ci(tn, tn + fp)
    ppv_lo,  ppv_hi  = wilson_ci(tp, tp + fp)
    npv_lo,  npv_hi  = wilson_ci(tn, tn + fn)
    acc_lo,  acc_hi  = wilson_ci(tp + tn, n)
    # F1 = 2TP/(2TP+FP+FN): Wilson CI with numerator=2TP, denominator=2TP+FP+FN
    f1_denom = 2 * tp + fp + fn
    f1_lo, f1_hi = (wilson_ci(2 * tp, f1_denom) if f1_denom > 0 else (np.nan, np.nan))
    kap_lo, kap_hi = kappa_ci(all_true, all_pred)

    micro_rows.append({
        'Source': src,
        'Sensitivity': round(sens, 4) if not np.isnan(sens) else np.nan,
        'Sensitivity_lo': sens_lo, 'Sensitivity_hi': sens_hi,
        'Specificity': round(spec, 4) if not np.isnan(spec) else np.nan,
        'Specificity_lo': spec_lo, 'Specificity_hi': spec_hi,
        'PPV':  round(ppv, 4) if not np.isnan(ppv) else np.nan,
        'PPV_lo':  ppv_lo,  'PPV_hi':  ppv_hi,
        'NPV':  round(npv, 4) if not np.isnan(npv) else np.nan,
        'NPV_lo':  npv_lo,  'NPV_hi':  npv_hi,
        'Accuracy': round(acc, 4),
        'Accuracy_lo': acc_lo, 'Accuracy_hi': acc_hi,
        'F1':  round(f1, 4),  'F1_lo':  f1_lo,  'F1_hi':  f1_hi,
        'Kappa': round(float(kap), 4), 'Kappa_lo': kap_lo, 'Kappa_hi': kap_hi,
    })

micro = pd.DataFrame(micro_rows).set_index('Source')

# --- Formatted display ---
print('Table 2a. Label-macro-averaged metrics (mean; 95% CI across 18 labels)\n')
rows_disp = []
for src in SOURCE_ORDER:
    row = {'Source': src}
    for m in ['Sensitivity', 'Specificity', 'PPV', 'NPV', 'Accuracy', 'F1']:
        mean_v, lo, hi = macro_ci[src][m]
        row[m] = fmt_pct(mean_v, lo, hi)
    mean_v, lo, hi = macro_ci[src]['Kappa']
    row['Kappa'] = fmt_kappa(mean_v, lo, hi)
    rows_disp.append(row)
disp_macro = pd.DataFrame(rows_disp).set_index('Source')
print(disp_macro.to_string())
print()

print('Table 2b. Micro-averaged metrics (TP/FP/FN pooled across 18 labels)\n')
rows_disp = []
for src in SOURCE_ORDER:
    row = {'Source': src}
    for m in ['Sensitivity', 'Specificity', 'PPV', 'NPV', 'Accuracy', 'F1']:
        row[m] = fmt_pct(
            micro.loc[src, m],
            micro.loc[src, f'{m}_lo'],
            micro.loc[src, f'{m}_hi'],
        )
    row['Kappa'] = fmt_kappa(
        micro.loc[src, 'Kappa'],
        micro.loc[src, 'Kappa_lo'],
        micro.loc[src, 'Kappa_hi'],
    )
    rows_disp.append(row)
disp_micro = pd.DataFrame(rows_disp).set_index('Source')
print(disp_micro.to_string())

## 5. Save Tables

In [ ]:
metrics_df.to_csv(TABLES_DIR / 'model_vs_human_per_label.csv', index=False)
label_macro.to_csv(TABLES_DIR / 'model_vs_human_label_macro.csv')
micro[METRIC_COLS].to_csv(TABLES_DIR / 'model_vs_human_micro.csv')
disp_macro.to_csv(TABLES_DIR / 'model_vs_human_label_macro_formatted.csv')
disp_micro.to_csv(TABLES_DIR / 'model_vs_human_micro_formatted.csv')
print('Saved: model_vs_human_per_label.csv')
print('Saved: model_vs_human_label_macro.csv')
print('Saved: model_vs_human_micro.csv')
print('Saved: model_vs_human_label_macro_formatted.csv')
print('Saved: model_vs_human_micro_formatted.csv')

## 6. Figures

### Fig 1 — F1 heatmap (labels × sources)

In [ ]:
# for metric, title, fname in [
#     ('F1',    'F1 score vs human annotations',  'mvh_fig1_f1_heatmap.png'),
#     ('Kappa', "Cohen's κ vs human annotations", 'mvh_fig2_kappa_heatmap.png'),
# ]:
#     pivot = metrics_df.pivot(index='Label', columns='Source', values=metric)[SOURCE_ORDER]
#     pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

#     # Annotation: XX.X% format (pandas >= 2.1 uses .map instead of .applymap)
#     annot = pivot.map(lambda v: f'{v * 100:.1f}%' if not np.isnan(v) else '')

#     fig, ax = plt.subplots(figsize=(len(SOURCE_ORDER) * 2.4 + 2, 7))
#     sns.heatmap(
#         pivot.astype(float),
#         annot=annot, fmt='', cmap='RdYlGn',
#         vmin=0, vmax=1, linewidths=0.4, linecolor='white',
#         annot_kws={'size': 9},
#         cbar_kws={'label': metric, 'shrink': 0.7},
#         ax=ax,
#     )
#     ax.set_title(title, pad=10)
#     ax.tick_params(axis='x', rotation=20, labelsize=9)
#     ax.tick_params(axis='y', rotation=0,  labelsize=8)
#     ax.set_xlabel('')
#     ax.set_ylabel('')
#     fig.tight_layout()
#     plt.savefig(FIGURES_DIR / fname, bbox_inches='tight', dpi=200)
#     plt.show()
#     print(f'Saved: {fname}')

# # --- Combined figure (F1 + Kappa stacked vertically) ---
# pivots = {}
# annots = {}
# for metric in ['F1', 'Kappa']:
#     p = metrics_df.pivot(index='Label', columns='Source', values=metric)[SOURCE_ORDER]
#     p = p.loc[p.mean(axis=1).sort_values(ascending=False).index]
#     pivots[metric] = p
#     annots[metric] = p.map(lambda v: f'{v * 100:.1f}%' if not np.isnan(v) else '')

# fig, axes = plt.subplots(2, 1, figsize=(len(SOURCE_ORDER) * 2.4 + 2, 14))
# for ax, metric, title in zip(
#     axes,
#     ['F1', 'Kappa'],
#     ['F1 score vs human annotations', "Cohen's κ vs human annotations"],
# ):
#     sns.heatmap(
#         pivots[metric].astype(float),
#         annot=annots[metric], fmt='', cmap='RdYlGn',
#         vmin=0, vmax=1, linewidths=0.4, linecolor='white',
#         annot_kws={'size': 9},
#         cbar_kws={'label': metric, 'shrink': 0.7},
#         ax=ax,
#     )
#     ax.set_title(title, pad=10)
#     ax.tick_params(axis='x', rotation=20, labelsize=9)
#     ax.tick_params(axis='y', rotation=0,  labelsize=8)
#     ax.set_xlabel('')
#     ax.set_ylabel('')

# fig.tight_layout()
# combined_fname = 'mvh_fig1_2_combined_heatmap.png'
# plt.savefig(FIGURES_DIR / combined_fname, bbox_inches='tight', dpi=200)
# plt.show()
# print(f'Saved: {combined_fname}')


In [ ]:
for metric, title, fname in [
    ('F1',    'F1 score vs human annotations',  'mvh_fig1_f1_heatmap.png'),
    ('Kappa', "Cohen's κ vs human annotations", 'mvh_fig2_kappa_heatmap.png'),
]:
    pivot = metrics_df.pivot(index='Label', columns='Source', values=metric)[SOURCE_ORDER]
    pivot = pivot.loc[pivot['CT-RATE (original)'].sort_values(ascending=False).index]

    annot = pivot.map(lambda v: f'{v * 100:.1f}%' if not np.isnan(v) else '')

    fig, ax = plt.subplots(figsize=(len(SOURCE_ORDER) * 2.4 + 2, 7))
    sns.heatmap(
        pivot.astype(float),
        annot=annot, fmt='', cmap='RdYlGn',
        vmin=0, vmax=1, linewidths=0.4, linecolor='white',
        annot_kws={'size': 9},
        #cbar_kws={'label': metric, 'shrink': 0.7},
        cbar_kws={'label': metric, 'shrink': 0.7, 'format': lambda x, _: f'{x*100:.0f}%'},
        ax=ax,
    )
    ax.set_title(title, pad=10)
    ax.tick_params(axis='x', rotation=20, labelsize=9)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)
    ax.set_xlabel('')
    ax.set_ylabel('')
    fig.tight_layout()
    plt.savefig(FIGURES_DIR / fname, bbox_inches='tight', dpi=200)
    plt.show()
    print(f'Saved: {fname}')

# --- Combined figure (F1 + Kappa stacked vertically) ---
pivots = {}
annots = {}
for metric in ['F1', 'Kappa']:
    p = metrics_df.pivot(index='Label', columns='Source', values=metric)[SOURCE_ORDER]
    p = p.loc[p['CT-RATE (original)'].sort_values(ascending=False).index]
    pivots[metric] = p
    annots[metric] = p.map(lambda v: f'{v * 100:.1f}%' if not np.isnan(v) else '')

fig, axes = plt.subplots(2, 1, figsize=(len(SOURCE_ORDER) * 2.4 + 2, 14))

for ax, metric, title in zip(
    axes,
    ['F1', 'Kappa'],
    ['F1 score vs human annotations', "Cohen's κ vs human annotations"],
):
    sns.heatmap(
        pivots[metric].astype(float),
        annot=annots[metric], fmt='', cmap='RdYlGn',
        vmin=0, vmax=1, linewidths=0.4, linecolor='white',
        annot_kws={'size': 9},
        #cbar_kws={'label': metric, 'shrink': 0.7},
        cbar_kws={'label': metric, 'shrink': 0.7, 'format': lambda x, _: f'{x*100:.0f}%'},
        ax=ax,
    )
    ax.set_title(title, pad=10)
    ax.tick_params(axis='x', rotation=20, labelsize=9)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)
    ax.set_xlabel('')
    ax.set_ylabel('')

fig.tight_layout()
combined_fname = 'mvh_fig1_2_combined_heatmap.png'
plt.savefig(FIGURES_DIR / combined_fname, bbox_inches='tight', dpi=200)
plt.show()
print(f'Saved: {combined_fname}')

### Fig 3 — Macro-metric bar chart

In [ ]:
display_metrics = ['Sensitivity', 'Specificity', 'PPV', 'NPV', 'Accuracy', 'F1', 'Kappa']

x = np.arange(len(display_metrics))
n_src = len(SOURCE_ORDER)
w = 0.7 / n_src

for avg_name, avg_df in [('Label-macro', label_macro), ('Micro', micro)]:
    fig, ax = plt.subplots(figsize=(11, 4.5))
    for i, src in enumerate(SOURCE_ORDER):
        vals = avg_df.loc[src, display_metrics].values.astype(float)
        ax.bar(
            x + (i - (n_src - 1) / 2) * w, vals,
            width=w, label=src, color=SOURCE_COLORS[src], alpha=0.85,
        )
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace('_', ' ') for m in display_metrics], fontsize=10)
    ax.set_ylim(0, 1.09)
    ax.set_ylabel('Score')
    ax.set_title(f'Performance vs human annotations — {avg_name} averaging')
    ax.legend(loc='lower right')
    ax.yaxis.grid(True, linestyle='--', alpha=0.4)
    ax.set_axisbelow(True)
    fig.tight_layout()
    fname = f"mvh_fig3_{'label_macro' if avg_name == 'Label-macro' else 'micro'}_bar.png"
    plt.savefig(FIGURES_DIR / fname, bbox_inches='tight', dpi=200)
    plt.show()
    print(f'Saved: {fname}')

### Fig 4 — Sensitivity vs Specificity scatter per label

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for src in SOURCE_ORDER:
    sub = metrics_df[metrics_df['Source'] == src]
    ax.scatter(
        sub['Specificity'], sub['Sensitivity'],
        s=70, color=SOURCE_COLORS[src], marker=SOURCE_MARKERS[src],
        alpha=0.8, label=src, edgecolors='grey', linewidths=0.4,
    )

# Annotate labels with mean F1 < 0.5 across sources
low_f1 = metrics_df.groupby('Label')['F1'].mean()
low_f1_labels = low_f1[low_f1 < 0.5].index.tolist()
ref_src = metrics_df[metrics_df['Source'] == SOURCE_ORDER[0]].set_index('Label')
for lbl in low_f1_labels:
    if lbl in ref_src.index:
        ax.annotate(
            lbl, (ref_src.loc[lbl, 'Specificity'], ref_src.loc[lbl, 'Sensitivity']),
            fontsize=7, ha='left', va='bottom',
            xytext=(4, 3), textcoords='offset points',
        )

ax.axhline(0.8, color='grey', linestyle='--', linewidth=0.8, alpha=0.5)
ax.axvline(0.8, color='grey', linestyle='--', linewidth=0.8, alpha=0.5)
ax.set_xlabel('Specificity')
ax.set_ylabel('Sensitivity')
ax.set_xlim(-0.05, 1.08)
ax.set_ylim(-0.05, 1.08)
ax.set_title('Sensitivity vs Specificity per label (vs human annotations)')
ax.legend()

fig.tight_layout()
plt.savefig(FIGURES_DIR / 'mvh_fig4_sens_spec_scatter.png', bbox_inches='tight', dpi=200)
plt.show()
print('Saved: mvh_fig4_sens_spec_scatter.png')

### Fig 5 — Per-label F1 dot plot

In [ ]:
label_order_f1 = (
    metrics_df.groupby('Label')['F1'].mean()
    .sort_values(ascending=True).index.tolist()
)

y_pos = np.arange(len(label_order_f1))
offsets = np.linspace(-0.28, 0.28, len(SOURCE_ORDER))

fig, ax = plt.subplots(figsize=(7, 7))
for i, src in enumerate(SOURCE_ORDER):
    sub = metrics_df[metrics_df['Source'] == src].set_index('Label')
    f1_vals = [sub.loc[lbl, 'F1'] if lbl in sub.index else np.nan for lbl in label_order_f1]
    ax.scatter(
        f1_vals, y_pos + offsets[i],
        s=55, color=SOURCE_COLORS[src], marker=SOURCE_MARKERS[src],
        label=src, alpha=0.85, edgecolors='grey', linewidths=0.3,
    )

ax.axvline(0.8, color='grey', linestyle='--', linewidth=0.8, alpha=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(label_order_f1, fontsize=8)
ax.set_xlabel('F1 score')
ax.set_xlim(-0.05, 1.08)
ax.set_title('Per-label F1 score vs human annotations')
ax.legend(loc='lower right')
ax.xaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)

fig.tight_layout()
plt.savefig(FIGURES_DIR / 'mvh_fig5_f1_dotplot.png', bbox_inches='tight', dpi=200)
plt.show()
print('Saved: mvh_fig5_f1_dotplot.png')

### Fig 6 — Label prevalence: human vs all sources

In [ ]:
prev_rows = []
for col in LABEL_COLS:
    row = {'Label': col, 'Human (GT)': human_idx[col].astype(int).mean() * 100}
    for src, df in source_idx.items():
        row[src] = df[col].astype(int).mean() * 100
    prev_rows.append(row)

prev_df = pd.DataFrame(prev_rows).set_index('Label')
prev_df = prev_df.loc[prev_df['Human (GT)'].sort_values(ascending=False).index]

all_prev_cols = ['Human (GT)'] + SOURCE_ORDER
all_prev_colors = [plt.cm.tab10(0.7)] + [SOURCE_COLORS[s] for s in SOURCE_ORDER]

n_cols = len(all_prev_cols)
x = np.arange(len(LABEL_COLS))
w = 0.75 / n_cols

fig, ax = plt.subplots(figsize=(14, 4.5))
for i, (col, color) in enumerate(zip(all_prev_cols, all_prev_colors)):
    lw = 2 if col == 'Human (GT)' else 0
    ec = 'black' if col == 'Human (GT)' else None
    ax.bar(
        x + (i - (n_cols - 1) / 2) * w,
        prev_df[col].values,
        width=w, label=col, color=color,
        alpha=0.85, edgecolor=ec, linewidth=lw,
    )

ax.set_xticks(x)
ax.set_xticklabels(prev_df.index, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Prevalence (%)')
ax.set_title('Label prevalence: human annotations vs all sources')
ax.legend()
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)

fig.tight_layout()
plt.savefig(FIGURES_DIR / 'mvh_fig6_prevalence.png', bbox_inches='tight', dpi=200)
plt.show()
print('Saved: mvh_fig6_prevalence.png')